# B3 — LightGBM, MLP & Đặc trưng đồ thị trên Kaggle

1. **Add Input** → tìm dataset *IBM Transactions for Anti Money Laundering (AML)* (tác giả `ealtman2019`) → bấm **+**.
2. **Session options / Settings** → **Internet: On** (để tải code từ GitHub). Chọn GPU T4 nếu chạy thêm `mlp`, hoặc `None` nếu chỉ chạy `lgbm`.
3. **Run All** (hoặc **Save Version → Save & Run All** để chạy nền nếu chạy nhiều fold/seed).
4. Kết quả nằm trong `/kaggle/working/ket_qua.zip` ở tab **Output** (gồm `bang.csv`, `BAO_CAO.md`, `log.txt`, `kaggle_b0_kiem_tra.txt`).

In [ ]:
REPO = 'https://github.com/trantatdat06/NCKH2026.git'
NHANH = 'em2/b3-mo-hinh'

%cd /kaggle/working
!rm -rf /kaggle/working/NCKH2026
!git clone -q --depth 1 -b {NHANH} {REPO} /kaggle/working/NCKH2026
%cd /kaggle/working/NCKH2026
!git log --oneline -1

In [ ]:
# Kaggle đã có sẵn numpy, pandas, scikit-learn, torch, lightgbm. Chỉ cài phần còn thiếu.
!pip install -q opacus==1.6.0 torch-geometric pyarrow
!python -m pytest -q tests -p no:cacheprovider --ignore=tests/test_consumer.py

In [ ]:
import glob
RAW = sorted(glob.glob('/kaggle/input/**/HI-Small_Trans.csv', recursive=True))[0]
ACC = sorted(glob.glob('/kaggle/input/**/HI-Small_accounts.csv', recursive=True))[0]
print(RAW, ACC, sep='\n')

In [ ]:
# Tính 22 đặc trưng cơ bản một lần (~3-5 phút)
!python scripts/06_ibm_features.py --input {RAW} --view bank

In [ ]:
# Kiểm tra B0: phải ra đủ 3 dòng KHỚP trước khi chạy thí nghiệm
!python scripts/b0_chia_co_dinh.py --kiem-tra 2>&1 | tee results/kaggle_b0_kiem_tra.txt

In [ ]:
# ===== Chạy thí nghiệm B3 (LightGBM + MLP + đặc trưng đồ thị) =====
# Bước 1 (mặc định): Chạy thử trên fold 0, seed 42 (khoảng 15-25 phút)
!python scripts/b3_lightgbm.py --raw {RAW} --mo-hinh lgbm mlp --bo-dac-trung co_ban do_thi_fl do_thi_nuoc do_thi_global --folds 0 --seeds 42 --tag thu_b3_f0

# Bước 2 (chạy 5 fold cho LightGBM — rất nhanh, bỏ dấu # bên dưới nếu muốn chạy luôn):
# !python scripts/b3_lightgbm.py --raw {RAW} --mo-hinh lgbm --bo-dac-trung co_ban do_thi_fl do_thi_nuoc do_thi_global --folds 0 1 2 3 4 --seeds 42 --tag b3_lgbm_5fold_s42

# Bước 3 (chốt chính thức 3 seed x 5 fold — dùng Save & Run All chạy nền):
# !python scripts/b3_lightgbm.py --raw {RAW} --mo-hinh lgbm --bo-dac-trung co_ban do_thi_fl do_thi_nuoc do_thi_global --folds 0 1 2 3 4 --seeds 42 7 2027 --tag chot_lgbm
# !python scripts/b3_lightgbm.py --raw {RAW} --mo-hinh mlp --bo-dac-trung co_ban do_thi_fl do_thi_nuoc do_thi_global --folds 0 1 2 3 4 --seeds 42 7 2027 --tag chot_mlp

In [ ]:
# Nén CHỈ kết quả mới sinh trên Kaggle thành /kaggle/working/ket_qua.zip
!cd /kaggle/working/NCKH2026 && rm -f /kaggle/working/ket_qua.zip \
  && { git ls-files --others --exclude-standard results; git diff --name-only -- results; } \
  | grep -v '^results/_luu_tru/' | zip -q /kaggle/working/ket_qua.zip -@ \
  && unzip -l /kaggle/working/ket_qua.zip | tail -n 20